# Агент мониторинга биржевых новостей
Агент с websearch мониторинга новостей по акциям Мосбиржи: WebSearch Responses API для выборки публикаций, Object Storage для сохранения истории и последующей аналитики

# 1. Введение

## Назначение

В этом кукбуке демонстрируется, как создать **интеллектуальный мониторинг новостей по акциям Мосбиржи** с помощью:
- **Yandex WebSearch Responses API** — для поиска релевантных новостей о компаниях и акциях
- **AI Studio** — для отбора важных новостей и генерации дайджестов
- **Object Storage** — для хранения истории сигналов и аналитики

## Описание задачи

**Бизнес-задача:** Инвесторам, трейдерам и финансовым аналитикам требуется оперативный мониторинг новостей о компаниях на Мосбирже. Этот кукбук демонстрирует:
- Подписку на события по акциям через WebSearch API
- Отбор значимых новостей с помощью YandexGPT
- Генерацию структурированных дайджестов с рейтингом влияния
- Хранение истории в Object Storage для аналитики

**Основные сервисы Yandex Cloud:**
- 🤖 **AI Studio YandexGPT** — анализ новостей, определение релевантности и генерация дайджестов
- 💾 **Object Storage** — архивирование результатов и истории сигналов
- 🔌+ 🔍 **Responses API c WebSearch Tool** — унифицированный интерфейс для вызова моделей с инструментами + поиск новостей о компаниях и акциях в реальном времени

***

## Ожидаемый результат

После выполнения этого ноутбука вы сможете:
1. ✅ Создать агента для мониторинга новостей по акциям Мосбиржи
2. ✅ Настроить фильтры поиска по компаниям и финансовым тикерам
3. ✅ Отбирать значимые новости с использованием критериев влияния
4. ✅ Генерировать структурированные дайджесты в формате JSON
5. ✅ Запускать мониторинг по расписанию через Cloud Functions
6. ✅ Сохранять историю в Object Storage для дальнейшего анализа
7. ✅ Пример встраимого виджета с агентом **(Взаимодействие с агентом через виджет возможно только в `google colab`)**

# 2. Архитектура решения

## Компоненты системы мониторинга

```
┌──────────────────────┐
│  Responses API       │
│  (YandexGPT)         │
│  - Анализ запроса    │
│  - Планирование      │
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│  WebSearch Tool      │
│  - Фильтры по тикерам│
│  - Поиск по ключевым │
│    словам (слияния,  │
│    дивиденды, отчеты)│
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│  Yandex Search       │
│  (Интернет)          │
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│       Agent          │
│  - Фильтрация        │
│  - Оценка релевант.  │
│  - Генерация digest  │
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│  Object Storage      │
│  - История событий   │
│  - JSON архивы       │
└──────────────────────┘
```

| Компонент | Назначение | Роли и права |
|--------|-----------|----------|
| **AI agent**  |   Вызов агента    |   `ai.assistants.editor`  |
| **Object Storage** | Хранение истории и архивов | `storage.configViewer`, `storage.editor` |
|   **YandexGPT**   | Анализ и фильтрация новостей | `ai.languageModels.user`|

**Сервисный аккаунт** — это учетная запись, от имени которой приложения и автоматизированные сервисы могут управлять ресурсами `Yandex Cloud`. В отличие от обычных пользовательских аккаунтов, сервисные аккаунты используются для программного доступа к ресурсам и не требуют браузерной аутентификации.

## Описание ролей и их области действия:
- `ai.assistants.editor` — *Область действия: каталог и вложенные ресурсы.* Эта роль предоставляет полный доступ к **AI-агентам Yandex AI Studio** и позволяет сервисному аккаунту создавать, настраивать и использовать агентов для выполнения сложных задач.

- `storage.editor` — *Область действия: каталог и вложенные ресурсы, или отдельный бакет (если назначена на конкретный бакет в консоли управления).* Эта роль дает полный доступ к объектам в хранилище **Yandex Object Storage**. Она позволяет создавать и удалять бакеты, загружать и скачивать объекты, управлять конфигурацией хранилища

- `ai.languageModels.user` — *Область действия: каталог и вложенные ресурсы.* Это минимальная роль для работы с моделями генерации текста Yandex AI Studio (YandexGPT). Она позволяет сервисному аккаунту отправлять запросы к моделям для генерации текста, создания векторных представлений текста и выполнения классификации.

- `storage.configViewer` [опционально] — *Область действия: организация, облако, каталог или отдельный бакет (роли на высших уровнях действуют на вложенные ресурсы; роль также можно назначить на конкретный бакет в консоли управления).*
Эта роль позволяет только просматривать конфигурацию и настройки бакетов, но не дает доступа к самим данным. Специалист с этой ролью может видеть, как настроено хранилище, но не сможет читать или изменять данные внутри объектов.

## Как назначить роль сервисному аккаунту через консоль Yandex Cloud:
1. Откройте сервис Identity and Access Management — перейдите в консоль управления и выберите сервис IAM из списка сервисов.

2. Выберите облако или каталог — нажмите селектор ресурса на панели сверху и выберите облако или каталог, к которому вы хотите предоставить доступ.

3. Перейдите на вкладку "Права доступа" — откройте раздел управления доступом для выбранного ресурса.

4. Нажмите "Настроить доступ" — откроется окно, где вы сможете добавить новые права доступа.

5. Выберите раздел "Сервисные аккаунты" — в появившемся окне нужно указать тип субъекта (получателя прав).

6. Найдите нужный сервисный аккаунт — выберите сервисный аккаунт из списка или создайте новый.

7. Добавьте роли — нажмите кнопку "Добавить роль" и выберите требуемые роли из списка. Вы можете назначить несколько ролей одному сервисному аккаунту.

8. Сохраните изменения — нажмите кнопку "Сохранить", чтобы применить назначенные роли.

##  3. Подготовка окружения

Эта ячейка подготавливает параметры и клиент для работы с сервисами Yandex Cloud:
- **FOLDER_ID** — идентификатор каталога для ресурсов: https://cloud.yandex.ru/docs/resource-manager/operations/folder/get-id
- **API_KEY** — апи ключ для вызовова API: https://yandex.cloud/ru/docs/iam/concepts/authorization/api-key
- **S3_ENDPOINT**, **YC_ACCESS_KEY/YC_SECRET_KEY**, **BUCKET_NAME** — настройки доступа к Object Storage (S3-совместимому) для хранения видео/аудио/транскриптов: https://cloud.yandex.ru/docs/storage/s3/
- Инициализация клиента s3 через boto3 с кастомным endpoint Yandex Object Storage: https://yandex.cloud/ru/docs/storage/tools/boto

Также есть возможность использования **IAM-токена** вместо **API_KEY**:  
- **IAM_TOKEN** — краткоживущий токен аутентификации в Yandex Cloud (до 12 часов). Получите его одним из способов:

    - Через Yandex Cloud CLI: `yc iam create-token` и сохраните значение в переменную окружения **IAM_TOKEN**;

    - Или обменяв OAuth-токен/учетные данные сервисного аккаунта на **IAM-токен** с помощью метода `IamToken.create`.

    - Токен необходимо периодически обновлять и передавать в заголовке Authorization: Bearer <IAM_TOKEN> при вызовах API.

Необходимо указать следующие параметры:

In [1]:
# Установка необходимых зависимостей
!pip install openai python-dotenv boto3 requests pydantic

In [2]:
import json
import os
import re
from datetime import datetime, timezone
from typing import Any, Callable, Dict, List, Literal, Optional, Tuple

import boto3
import openai
from dotenv import find_dotenv, load_dotenv

In [3]:
load_dotenv(find_dotenv())

S3_BUCKET = os.getenv("BUCKET_NAME", "YOUR_BUCKET_NAME")
S3_REGION = "ru-central1"


s3 = boto3.client(
"s3",
endpoint_url=f"https://storage.yandexcloud.net",
aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID", "YOUR_AWS_ACCESS_ID"),
aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY", "YOUR_AWS_SECRET_ACCESS_KEY"),
)
YANDEX_CLOUD_FOLDER = os.getenv("YANDEX_CLOUD_FOLDER", "YOUR_YANDEX_CLOUD_FOLDER") # Вставьте сюда свой FOLDER_ID
YANDEX_CLOUD_API_KEY = os.getenv("YANDEX_CLOUD_API_KEY", "YOUR_YANDEX_CLOUD_API_KEY") # Вставьте сюда свой API_KEY

# 3.1. Вызов Yandex LLM через OpenAI-совместимое API

Эта ячейка отправляет промпт в LLM через OpenAI-совместимый endpoint:
- Идентификатор модели задаётся URI вида `gpt://{folder_id}/{model_id}/latest`; передаём messages `(system/user)`, управляем параметрами `max_tokens` и `temperature`.
- Возвращается текст ответа модели, который ожидается в формате строгого JSON согласно заданному PROMPT.

Документация:
- AI STUDIO — обзор: https://yandex.cloud/ru/docs/ai-studio/quickstart/yandexgpt
- OpenAI-совместимое API (использование SDK): https://yandex.cloud/ru/docs/foundation-models/concepts/openai-compatibility


In [4]:
YANDEX_MODEL = "yandexgpt" # Вы можете подключить любую модель https://yandex.cloud/en/docs/ai-studio/concepts/generation/models

# Инициализация агента
# Пример агента взят с https://github.com/yandex-ai-studio/yandex-ai-studio-api-examples/blob/main/responses/web_tool.py
try:
    client = openai.OpenAI(
        api_key=YANDEX_CLOUD_API_KEY,
        base_url="https://rest-assistant.api.cloud.yandex.net/v1",
        project=YANDEX_CLOUD_FOLDER
    )

    # формирование запроса
    response = client.responses.create(
        model=f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_MODEL}",
        input=[
            {"role": "system", "content": "Отвечай кратко и на русском"},
            {"role": "user", "content": "Какая погода сейчас в Москве."},
        ],
        tools=[
            {
                    "type": "web_search",
                    "filters": {
                        "allowed_domains": [
                            "..." # поиск без ограничений по доменам
                        ],
                        "user_location": {
                            "region": "213", # регион поиска
                        }
                    }
                }
        ],
        temperature=0,
        top_p=1,
        max_output_tokens=1000
    )

    # Текст ответа
    print("Текст ответа:")
    print(response.output_text)
    print("\n" + "=" * 100 + "\n")

    # Полный ответ
    print("Структура ответа (JSON):")
    print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

except Exception as e:
    print(f"Ошибка инициализации и вызова агента: {e}")



Текст ответа:
Сейчас в Москве температура воздуха составляет около -9°C, ощущается как -12°C. Погода облачная, идёт снег. Влажность воздуха составляет 76%, а атмосферное давление — 748 мм рт. ст.


Структура ответа (JSON):
{
  "id": "d5ee36d8-a593-4464-86b7-4fe3c15b4978",
  "created_at": 1772096559.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt://b1g3bmsq14tvqeggdhhc/yandexgpt",
  "object": "response",
  "output": [
    {
      "id": "8438f01b-a2f7-4854-944a-7d898bd8f513",
      "action": {
        "query": "{\"query\":\"какая погода сейчас в Москве\"}",
        "type": "search",
        "queries": null,
        "sources": null,
        "valid": true
      },
      "status": "completed",
      "type": "web_search_call",
      "valid": true
    },
    {
      "id": "30191e14-3833-4595-b0e3-3f16718a1d04",
      "content": [
        {
          "annotations": [
            {
              "end_index": 0,
              "start_ind

# 4. Логика приложения: Агент мониторинга новостей

## Описание агента

Агент использует **Responses API** с инструментом `web_search` для:
1. **Поиска новостей** по тикерам акций и финансовым ключевым словам
2. **Анализа релевантности** найденных новостей (фильтрация spam, неактуальных)
3. **Оценки влияния** на цену акции (значимость события)
4. **Генерации структурированного дайджеста** в формате JSON
5. **Сохранения результатов** в Object Storage с временными метками

## 4.1 Определение инструмента WebSearch для поиска новостей

Инструмент настроен для:
- Поиска информации о компаниях и акциях в русскоязычном интернете
- Фильтрации по регионам (Москва и Россия)
- Поддержки ключевых слов финансовых событий (IPO, дивиденды, слияния, отчеты)

In [5]:
# 📍 Целевые домены для поиска (ВАЖНО: указывает, на каких сайтах искать)
SEARCH_DOMAINS = [
    'moex.com',
    'cbonds.ru',
    'smart-lab.ru',
    'tbank.ru',
    'yandex.ru'
]

# Регион пользователя (код Яндекса)
USER_REGION = '213'  # Москва
# Таблица регионов (https://yandex.ru/dev/xml/doc/ru/reference/regions)

In [6]:
# Определение инструмента WebSearch для мониторинга новостей
WEB_SEARCH_TOOL = {
    "type": "web_search",
    "filters": {
        # Поиск проводится по всему интернету (без ограничений по доменам)
        # для получения всех новостей о компаниях
        "allowed_domains": SEARCH_DOMAINS,  # Пусто = поиск везде
        "user_location": {
            "region": USER_REGION  # Москва (код Яндекса)
        }
    }
}

print(json.dumps(WEB_SEARCH_TOOL, indent=2, ensure_ascii=False))

{
  "type": "web_search",
  "filters": {
    "allowed_domains": [
      "moex.com",
      "cbonds.ru",
      "smart-lab.ru",
      "tbank.ru",
      "yandex.ru"
    ],
    "user_location": {
      "region": "213"
    }
  }
}


## 4.2 Определение структурированного вывода (JSON Schema)

Агент возвращает структурированный JSON с информацией о найденных новостях и их анализе.

In [7]:
from pydantic import BaseModel, Field
#---------------------------Модели Structured Output для оценки влияния---------------------

class Influence(BaseModel):
    reasoning: str = Field(..., description="Краткое обоснование")
    sentiment: Literal["Позитивное", "Негативное", "Нейтральное"] = Field(..., description="Тональность")

class InfluenceAssessment(BaseModel):
    """Модель для парсинга ответа нейросети"""
    title: str = Field(..., description="Заголовок новости")
    summary: str = Field(..., description="Суть в 1 предложении")
    influence: Influence

# #----------------------------Модели для формирования ответа агента--------------------------------

class NewsDigest(BaseModel):
    title: str   # Заголовок новости (кратко описывает суть материала)
    urls: List[str]    # Ссылка на источник
    summary: str    # Краткое содержание новости (1–3 предложения или сжатый пересказ)
    influence: Influence  # ИСПРАВЛЕНО: здесь должен быть Influence, а не InfluenceAssessment

class StockDigest(BaseModel):
    """Сводка по акции"""
    ticker: str   # Тикер акции на бирже (например, AAPL, SBER)
    news_digest: Optional[NewsDigest] = None   # Список объектов NewsDigest с новостями по данной акции

## 4.3 Реализация агента мониторинга

Агент принимает список тикеров и ищет актуальные новости для каждого.

Функция парсинга новости для преобразования в словари

In [8]:
def extract_urls_from_text(text: str) -> List[str]:
    """Извлекает URL из текста в формате Markdown [title](url)"""
    pattern = r'\[.+?\]\((https?://[^\)]+)\)'
    urls = re.findall(pattern, text)
    return list(set(urls))

def parse_response_text(response) -> tuple[str, List[str]]:
    """
    Извлекает текст и URL из ответа YandexGPT.
    Возвращает: (текст, список_url)
    """
    text = ""
    urls = []

    for item in getattr(response, "output", []):
        if not hasattr(item, "content"):
            continue

        for block in item.content:
            # Извлекаем текст
            if hasattr(block, "text") and block.text:
                text += block.text

            # Извлекаем URL из аннотаций
            if hasattr(block, "annotations"):
                for ann in block.annotations:
                    if hasattr(ann, "url") and ann.url:
                        urls.append(ann.url)

    # Дополнительно ищем URL в тексте
    urls.extend(extract_urls_from_text(text))

    return text, list(set(urls))

In [9]:

def analyze_news_influence(client: openai.OpenAI, model_uri: str, news_text: str, ticker: str, verbose: bool = True) -> List[NewsDigest]:
    """
    Второй вызов модели: превращает сырой текст сводки в список структурированных новостей с оценками.
    """
    if not news_text or len(news_text) < 10:
        return []

    system_prompt = f"""
    Ты — финансовый аналитик.
    Твоя задача — извлечь из текста новости о компании {ticker} и нужно провести сентимент анализ новости
    """

    try:
        if verbose:
            print(f"Анализирую влияние для {ticker}...")

        response = client.responses.parse(
            model=model_uri,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Проанализируй текст:\n{news_text[:10000]}"},
            ],
            temperature=0,
            top_p=1,
            max_output_tokens=2000,
            text_format=InfluenceAssessment
        )

        if hasattr(response.output, 'refusal') and response.output.refusal:
            print(f"{ticker}: Модель отказалась: {response.output.refusal}")

        return response.output_parsed

    except Exception as e:
        print(f"Ошибка при анализе влияния: {e}")
        if verbose:
            import traceback
            traceback.print_exc()
        return []

In [10]:
def create_moex_news_agent(
    client: openai.OpenAI,
    model_uri: str,
    tools: List[Dict[str, Any]],
    system_prompt: str,
    temperature=0,
    top_p=1,
    max_output_tokens=4000,
    ) -> Callable[..., StockDigest]:
    """Создаёт агента для мониторинга новостей по акциям Мосбиржи"""

    def monitor_stock(user_query: str = "найди всю актуальную информацию по тикеру акции на мосбирже", ticker: str = "", verbose: bool = False) -> StockDigest:
        """Мониторит новости по одной акции"""

        user_query += f"{ticker}"
        if verbose:
            print(f"\nМониторю: {ticker}")

        try:
            response = client.responses.create(
                model=model_uri,
                input=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_query},
                ],
                tools=tools,
                temperature=temperature,
                max_output_tokens=max_output_tokens,
            )

            summary, sources = parse_response_text(response)
            if not summary:
              print(f"{ticker}: WebSearch вернул пустой summary")
              print(f"Response output: {response.output}")


            if summary:
                influence_response = analyze_news_influence(client, model_uri, summary, ticker)

                # ФОРМИРУЕМ ЕДИНУЮ СВОДКУ
                # Передаем все источники списком urls=sources
                digest_obj = NewsDigest(
                    title=f"Сводка новостей по {ticker}",
                    summary=summary,
                    urls=sources,  # Весь список ссылок сразу
                    influence=influence_response.influence
                )
            else:
                digest_obj = None

            if verbose:
                print(f"Новости: {digest_obj}")

            return StockDigest(
                ticker=ticker,
                news_digest=digest_obj,
            )

        except Exception as e:
            print(f"Ошибка: {str(e)}")
            return StockDigest(
                ticker=ticker,
                news_digest=None,
            )

    return monitor_stock

## 4.4 Запуск агента

In [11]:

def run_monitor(
    monitor_agent: Callable[..., StockDigest],
    user_query: str,
    ticker: str,
    verbose: bool = False
    ) -> StockDigest:
    """Запуск агента без сохранения в S3"""
    stock_digest = monitor_agent(
        user_query=user_query,
        ticker=ticker,
        verbose=verbose
    )
    return stock_digest


def build_result_payload(ticker: str, stock_digest: StockDigest):
    """
    Приводит результат агента (StockDigest) к JSON-совместимому виду для фронта/виджета.
    """
    # Считаем количество источников как "найденные материалы"
    count = 0
    if stock_digest.news_digest and stock_digest.news_digest.urls:
        count = len(stock_digest.news_digest.urls)

    return {
        "ticker": ticker,
        "articles_found": count,
        "digest": stock_digest.model_dump() if hasattr(stock_digest, "model_dump") else stock_digest.__dict__,
    }

## 4.5 Сохранение данных в S3 (ОПЦИОНАЛЬНО)

In [ ]:
def save_agent_result_to_s3(ticker: str, stock_digest: StockDigest, s3_client: Any, bucket_name: str) -> Dict[str, Any]:
    """
    Сохранить результаты агента мониторинга в Object Storage (S3)
    Адаптировано под единую сводку (NewsDigest)
    """
    try:
        # Используем UTC время
        current_utc_time = datetime.now(timezone.utc)
        timestamp = current_utc_time.isoformat()

        # Извлекаем сводку, если она есть
        digest = stock_digest.news_digest
        digest_data = None
        sources_count = 0

        if digest:
            sources_count = len(digest.urls)
            digest_data = {
                "title": digest.title,
                "summary": digest.summary,
                "urls": digest.urls,
                "influence": {
                    "reasoning": digest.influence.reasoning,
                    "sentiment": digest.influence.sentiment
                }
            }

        # Создать объект данных для сохранения
        data = {
            "timestamp": timestamp,
            "ticker": ticker,
            "sources_count": sources_count, # Количество источников
            "news_digest": digest_data      # Единая сводка
        }

        # Правильный формат пути в S3
        date_path = current_utc_time.strftime('%Y/%m/%d')
        time_path = current_utc_time.strftime('%H_%M_%S_%f')
        s3_key = f"news_monitoring/{ticker}/{date_path}/digest_{time_path}.json"

        print(f"Подготовка файла: {s3_key}")

        # Явная сериализация в JSON
        json_body = json.dumps(data, ensure_ascii=False, indent=2)

        print(f"Отправка в S3 ({len(json_body)} байт)...")

        # Загрузка в S3
        s3_client.put_object(
            Bucket=bucket_name,
            Key=s3_key,
            Body=json_body,
            ContentType='application/json; charset=utf-8'
        )

        print(f"✓ Сохранено в S3: s3://{bucket_name}/{s3_key}")

        return {
            "success": True,
            "s3_key": s3_key,
            "size": len(json_body),
            "timestamp": timestamp,
            # Для совместимости с виджетом возвращаем кол-во источников как saved
            "articles_saved": sources_count
        }

    except Exception as e:
        print(f"✗ Ошибка сохранения: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        return {
            "success": False,
            "error": str(e)
        }


def save_to_s3(
    ticker: str,
    stock_digest: StockDigest,
    s3_client: Any,
    bucket_name: str,
    enabled: bool
    ) -> Optional[Dict[str, Any]]:
  """Опциональный вызов save_agent_result_to_s3 для Html виджета"""
  if not enabled:
      return None
  return save_agent_result_to_s3(
      ticker=ticker,
      stock_digest=stock_digest,
      s3_client=s3_client,
      bucket_name=bucket_name
  )


# 5. Тестирование решения

## 5.1. Примеры системных промптов

Системный промпт определяет поведение агента и правила фильтрации новостей.

In [12]:
base_system_prompt = """
РОЛЬ И ЦЕЛЬ:
Ты — аналитический агент мониторинга новостей и публичных данных (рынок/компании/отрасли).
Твоя задача — находить актуальные публикации через web_search, извлекать факты, очищать шум
и формировать полезный аналитический вывод для принятия решений.

ПРИНЦИПЫ:
1. Фактичность: опирайся на источники из web_search. Не выдумывай данные.
2. Прозрачность: разделяй "факт" и "интерпретацию".
3. Проверяемость: всегда сохраняй ссылки на источники.
4. Релевантность: отбрасывай рекламу и дубли.
5. Язык: русский, деловой и нейтральный.

КАК ИСПОЛЬЗУЕШЬ WEB_SEARCH:
- Используй поиск для получения фактов.
- Поиск производится по ключевым словам
   - тикер или название компании (например, "Сбербанк SBER")
   - Финансовые события ("IPO", "дивиденды", "слияние", "отчет", "санкции") на сегодняшний день
   - Географические регионы (Россия, Мосбиржа, РТС)
- Если данных недостаточно — сообщи об этом.
- Всегда подтверждай ключевые тезисы ссылками.

БАЗОВЫЙ ФОРМАТ ОТВЕТА:
1. Короткий итог.
2. Ключевые факты (bullet points с ссылками).
3. Список источников.
"""

### Дополнения к системному промпту, ориентированные на определённые задачи

In [13]:

# --- МОДУЛИ: WEB SEARCH ANALYTICS ---
addon_external_analytics_trends = """
АНАЛИЗ НОВОСТЕЙ И ТРЕНДОВ:
Сгруппируй найденные материалы по темам. Для каждой группы укажи:
- Краткий тезис (что за событие)
- Почему это важно
- Подтверждающие ссылки
Выделяй изменения во времени ("появилось/усилилось") только при наличии дат.
"""

# --- МОДУЛИ: АКТУАЛЬНОСТЬ (RECENCY) ---
addon_recency_30d = """
АКТУАЛЬНОСТЬ: ПОСЛЕДНИЕ 30 ДНЕЙ (СТРОГО)
Фильтруй результаты: используй только материалы с датой публикации в рамках последних 30 дней.
Если дата не указана — помечай как "дата не подтверждена".
Если новостей за 30 дней нет — прямо сообщи об этом.
"""

addon_sources_official_only = """
ИСТОЧНИКИ: ТОЛЬКО ОФИЦИАЛЬНЫЕ
Используй: сайты бирж, регуляторов, пресс-релизы, разделы Investor Relations.
Игнорируй: блоги, форумы, слухи, мнения без подтверждения.
Если официальных данных нет — выдели блок "Неофициальные сигналы (требуют проверки)".
"""

# --- МОДУЛИ: ПОСТ-ОБРАБОТКА (POST-PROCESS) ---
addon_postprocess_news_summarization = """
СУММАРИЗАЦИЯ НОВОСТЕЙ:
Каждую новость сжимай до 1-2 предложений: "ЧТО случилось" + "ПОЧЕМУ это важно".
Ссылка на источник обязательна рядом с заголовком.
"""

addon_postprocess_exec_digest = """
ДАЙДЖЕСТ ДЛЯ РУКОВОДИТЕЛЯ:
Структура:
1. Executive summary (макс 6 строк)
2. Top-5 событий (Факт + Влияние + Ссылка)
3. Риски и возможности
4. Action items (что проверить)
Тон: максимально деловой, без воды.
"""

# --- МОДУЛИ: COMPLIANCE ---
addon_compliance_no_pii = """
ПЕРСОНАЛЬНЫЕ ДАННЫЕ (PII) — ЗАПРЕЩЕНЫ:
Не собирай и не выводи телефоны, адреса, ФИО частных лиц.
Если запрос на поиск человека — откажись, предложи поиск корпоративных данных.
"""

addon_compliance_fin_legal = """
ФИНАНСОВАЯ/ЮРИДИЧЕСКАЯ ИНФОРМАЦИЯ:
Ты даешь только справочную информацию, НЕ инвестиционные советы.
Добавляй дисклеймер: "Не является инвестиционной рекомендацией".
Юридические факты подавай с формулировкой "по данным источника".
"""

# --- МОДУЛИ: СТИЛЬ И ФОРМАТ ---
addon_style_official = """
СТИЛЬ: ОФИЦИАЛЬНЫЙ
Используй активный залог и точные глаголы.
Избегай оценочных эпитетов ("крах", "взлет") и кликбейта.
"""

addon_no_assumptions = """
ЗАПРЕТ ПРЕДПОЛОЖЕНИЙ:
Не делай выводов, не следующих из источников.
Любую интерпретацию помечай как "гипотеза" с обоснованием.
"""

addon_mandatory_citations = """
ОБЯЗАТЕЛЬНЫЕ ИСТОЧНИКИ:
Каждый факт должен иметь ссылку [url].
В конце ответа добавь раздел "## Источники" со списком всех ссылок.
"""

# Форматы вывода

addon_output_markdown = """
ФОРМАТ ВЫВОДА: MARKDOWN
Используй заголовки ##, списки и встроенные ссылки [текст](url).
"""

### Примеры сборки системных промптов

In [14]:
def build_system_prompt(task_type:str ="default"):
    """
    Собирает системный промпт из модулей под конкретную задачу
    """
    # Основа для всех агентов
    core_prompt = [
        base_system_prompt,
        addon_style_official,
        addon_mandatory_citations,
        addon_no_assumptions,
        addon_compliance_no_pii,
        addon_compliance_fin_legal
    ]

    if task_type == "digest_boss":
        # Дайджест для руководителя: 30 дней, строго официально, executive summary
        core_prompt.extend([
            addon_recency_30d,
            addon_sources_official_only,
            addon_postprocess_exec_digest,
        ])

    elif task_type == "market_research":
        # Исследование рынка: тренды, агрегация данных
        core_prompt.extend([
            addon_external_analytics_trends
        ])


    else:
        # Default fallback
        core_prompt.append(addon_output_markdown)

    return "\n\n".join(core_prompt)

## 5.2.1 Использование агента без сохранения данных

### Протестируем агента на примере нескольких акций.



Для использования веб-поиска лучше всего подходят модели `YandexGPT 5 Pro(yandexgpt)` и `YandexGPT 5 Lite(yandexgpt-lite)`
| Задача агента                            | Модель                           | Почему именно она                                                                                                                                          |
| ---------------------------------------- | -------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Глубокие ответы, много документов, RAG   | YandexGPT 5 Pro(yandexgpt)       | Лучше понимает сложные инструкции, точнее работает с внешними источниками и поддерживает до 32k токенов контекста, что критично при длинных веб‑сниппетах. |
| Быстрый/дешёвый FAQ‑бот, короткие ответы | YandexGPT 5 Lite(yandexgpt-lite) | Оптимизирована под низкую задержку и высокий TPS при схожем контекстном окне, поэтому выгоднее для массового web‑поискового ассистента.                    |

In [15]:
# Тестирование на примере акций
YANDEX_MODEL = "yandexgpt" # Вы можете подключить любую модель https://yandex.cloud/en/docs/ai-studio/concepts/generation/models


# Список тикеров Мосбиржи для мониторинга
STOCK = [
    "SBER",
    "MGNT",
    "YDEX"
]

# Создание агента
monitor_agent = create_moex_news_agent(
    client=client,
    model_uri=f"gpt://{YANDEX_CLOUD_FOLDER}/yandexgpt",
    tools=[WEB_SEARCH_TOOL],
    system_prompt=build_system_prompt("market_research"),
)

results = []
for ticker in STOCK:
  result = run_monitor(
      monitor_agent,
      user_query="Найди всю актуальную информацию за последние 7 дней по акции",
      ticker=ticker,
      verbose=True
  )
  results.append(result)


Мониторю: SBER
Анализирую влияние для SBER...
Новости: title='Сводка новостей по SBER' urls=['https://www.tbank.ru/invest/stocks/SBER/news/', 'https://smart-lab.ru/forum/news/SBER/', 'https://www.tbank.ru/invest/stocks/SBER/pulse/'] summary='### Короткий итог\nЗа последние 7 дней акции Сбербанка (SBER) показали умеренное повышение. Ожидается публикация отчета по МСФО за 2025 год, что может повлиять на динамику акций.\n\n### Ключевые факты\n- Чистая прибыль Сбербанка за январь 2026 года выросла на 21,7% по сравнению с аналогичным периодом прошлого года [1].\n- Банк планирует инвестировать 350 млрд рублей в генеративный ИИ в 2026 году [2].\n- Консенсус-прогноз по чистой прибыли Группы Сбер за 2025 год по МСФО составил 1,7 трлн рублей [3].\n- Отчетность Сбербанка за 2025 год будет опубликована 26 февраля [4].\n\n### Список источников\n1. https://www.tbank.ru/invest/stocks/SBER/news/\n2. https://smart-lab.ru/forum/news/SBER/\n3. https://www.tbank.ru/invest/stocks/SBER/pulse/' influence=In

## 5.2.2 Использование агента с сохранением данных в S3 (Опционально)

### Протестируем агента на примере нескольких акций.

In [ ]:
# Тестирование на примере акций
YANDEX_MODEL = "yandexgpt" # Вы можете подключить любую модель https://yandex.cloud/en/docs/ai-studio/concepts/generation/models


# Список тикеров Мосбиржи для мониторинга
STOCK = [
    "SBER",
    "MGNT",
    "YDEX"
]

# Создание агента
monitor_agent = create_moex_news_agent(
    client=client,
    model_uri=f"gpt://{YANDEX_CLOUD_FOLDER}/yandexgpt",
    tools=[WEB_SEARCH_TOOL],
    system_prompt=build_system_prompt("market_research"),
)

results = []
for ticker in STOCK:
    digest = run_monitor(
    monitor_agent=monitor_agent,
    user_query="Найди всю актуальную информацию за последние 7 дней по акции",
    ticker=ticker,
    verbose=False
    )

    s3_info = save_to_s3(
        ticker=ticker,
        stock_digest=digest,
        s3_client=s3,
        bucket_name=S3_BUCKET,
        enabled=True
    )
    results.append(digest)
    results.append(s3_info)



Анализирую влияние для SBER...
Подготовка файла: news_monitoring/SBER/2026/02/04/digest_20_19_42_807286.json
Отправка в S3 (1657 байт)...
✓ Сохранено в S3: s3://webagent-storage/news_monitoring/SBER/2026/02/04/digest_20_19_42_807286.json
Анализирую влияние для MGNT...
Подготовка файла: news_monitoring/MGNT/2026/02/04/digest_20_19_49_934802.json
Отправка в S3 (2067 байт)...
✓ Сохранено в S3: s3://webagent-storage/news_monitoring/MGNT/2026/02/04/digest_20_19_49_934802.json
Анализирую влияние для YDEX...
Подготовка файла: news_monitoring/YDEX/2026/02/04/digest_20_19_57_264600.json
Отправка в S3 (1937 байт)...
✓ Сохранено в S3: s3://webagent-storage/news_monitoring/YDEX/2026/02/04/digest_20_19_57_264600.json


HTML-виджет

In [ ]:
from IPython.display import HTML, display, JSON
import json
import os

WIDGET_HTML_PATH = "moex_widget.html"

if not os.path.exists(WIDGET_HTML_PATH):
    raise FileNotFoundError(f"Не найден {WIDGET_HTML_PATH} (abs: {os.path.abspath(WIDGET_HTML_PATH)})")

with open(WIDGET_HTML_PATH, "r", encoding="utf-8") as f:
    widget_html_content = f.read()

def check_s3_available():
    """Проверка наличия s3"""
    try:
        return (
            'save_to_s3' in globals() and
            callable(globals().get('save_to_s3')) and
            's3' in globals() and
            globals().get('s3') is not None and
            'S3_BUCKET' in globals() and
            globals().get('S3_BUCKET')
        )
    except Exception:
        return False

def moexagentcallback(payload: dict) -> str:
    tickers = (payload or {}).get("tickers") or []
    save_to_s3_flag = bool((payload or {}).get("save_to_s3", False))

    tickers = [str(t).upper().strip() for t in tickers if str(t).strip()]
    tickers = list(dict.fromkeys(tickers))

    s3_available = check_s3_available()

    results = []
    errors = []

    for ticker in tickers:
        try:
            stock_digest = run_monitor(
                monitor_agent=monitor_agent,
                user_query="Найди всю актуальную информацию за последние 7 дней по акции",
                ticker=ticker,
                verbose=False
            )

            base = build_result_payload(ticker=ticker, stock_digest=stock_digest)

            # Нормализация статей для виджета
            digest = base.get("digest") or {}
            arts = digest.get("articles") or []
            if not isinstance(arts, list):
                arts = []

            cleaned = []
            for a in arts:
                if not isinstance(a, dict):
                    continue
                infl = a.get("influence") or {}
                if not isinstance(infl, dict):
                    infl = {}
                cleaned.append({
                    "title": a.get("title", ""),
                    "summary": a.get("summary", ""),
                    "url": a.get("url", ""),
                    "influence": {
                        "reasoning": infl.get("reasoning", ""),
                        "sentiment": infl.get("sentiment", "")
                    }
                })

            base["digest"] = {**digest, "articles": cleaned}

            # Попытка сохранения в S3 только при наличии всех зависимостей
            s3_info = None
            if save_to_s3_flag:
                if s3_available:
                    try:
                        s3_info = globals()['save_to_s3'](
                            ticker=ticker,
                            stock_digest=stock_digest,
                            s3_client=globals()['s3'],
                            bucket_name=globals()['S3_BUCKET'],
                            enabled=True
                        )
                    except Exception as s3_error:
                        s3_info = {"error": f"S3 ошибка сохранения: {str(s3_error)}"}
                else:
                    s3_info = {"error": "S3 зависимости недоступны"}

            base["s3save"] = s3_info
            results.append(base)

        except Exception as e:
            errors.append(f"{ticker}: {str(e)}")

    return json.dumps({
        "results": results,
        "error": "; ".join(errors) if errors else None
    }, ensure_ascii=False)


try:
    from google.colab import output
    output.register_callback("moexagentcallback", moexagentcallback)
except Exception:
    print("Не Colab (или нет output.register_callback). Пропускаю регистрацию.")

display(HTML(widget_html_content))


# 6. Результаты и анализ

## Что мы достигли

✅ **Успешно создан агент для мониторинга новостей по акциям Мосбиржи**, который:
1. Ищет свежие новости о компаниях и событиях на акциях
2. Фильтрует результаты по релевантности
3. Генерирует структурированные дайджесты в JSON
5. Возвращает ссылки на исходные источники


# Как это работает

## Поток данных агента мониторинга

```
Пользователь: "Найди новости по акциям Мосбиржи"
         ↓
    Responses API (YandexGPT)
         ↓
   WebSearch Tool — поиск новостей
         ↓
   Поиск на Yandex Search
    (moex.com, cbonds.ru, smart-lab.ru, tbank.ru, yandex.ru)
         ↓
   YandexGPT обрабатывает результаты
    - Фильтрует по релевантности
    - Анализирует новости
    - Генерирует дайджест
         ↓
   StockDigest — структурированный результат
    ├─ Список найденных новостей
    ├─ Источники
    ├─ Общее резюме
    └─ Статистика
         ↓
    Добавление дайджеста в S3 в формате JSON
Результат: Структурированный JSON с новостями и добавление данных в S3 хранилище
```

## Ключевые параметры

| Параметр | Значение | Назначение |
|----------|----------|----------|
| `temperature` | 0 | Низкое значение для точности и фактичности |
| `top_p` | 1 | отбор минимального множества вероятных токенов при генерации |
| `max_output_tokens` | 4000 | Достаточно для подробных ответов |
| `allowed_domains` | [moex.com, cbonds.ru, smart-lab.ru, tbank.ru] | Ограничение поиска только финансовыми источниками |
| `user_region` | 213 | Москва — для релевантности результатов |
| `model` | yandexgpt | YandexGPT 5 Pro для глубокого анализа |
| `tools` | web_search | Инструмент для поиска в интернете |

## Основные компоненты

### 1. WebSearch Tool
Определяет параметры поиска:
- **allowed_domains** — сайты, на которых искать (Мосбиржа, финансовые порталы)
- **user_location** — регион для релевантности (код Яндекса 213 = Москва)

### 2. YandexGPT (Responses API)
Обрабатывает поиск и анализирует результаты:
- Формирует запросы к WebSearch Tool
- Анализирует найденные новости
- Генерирует структурированный дайджест
- Определяет тональность (positive/neutral/negative)

### 3. StockDigest (структура вывода)
```json
{
"timestamp": "2025-12-12T01:53:06.221345",
  "ticker": "YDEX",
  "articles_count": 2,
  "articles": [
    {
      "title": "Сводка новостей по YNDX #1",
      "source": "https://www.tbank.ru/invest/stocks/YNDX@US/news/",
      "summary": "### Краткое резюме\n\nНа основании последних данных, значимых новостей, которые могли бы существенно повлиять на цену акций YNDX (Яндекс), не обнаружено. Однако есть информация о динамике курса акций Yandex NV на Московской бирже за последние дни.\n\n### Список обнаруженных новостей\n\n1. **Курс акций Яндекс YNDX в рублях на MOEX**\n   - **Источник:** [Яндекс.Новости](https://news.yandex.ru/news/quotes/43.html)\n   - **Краткое содержание:** Представлена динамика курса акций Yandex NV (Яндекс) в рублях на Московской бирже за последние 10 дней.\n   - **Релевантность:** 1 (высокая)\n\n2. **Информация о порядке приобретения акций Yandex N.V.**\n   - **Источник:** [Московская Биржа](https://www.moex.com/n69465)\n   - **Краткое содержание:** Описание условий и порядка приобретения акций обыкновенных класса А Yandex N.V. с одновременной продажей акций МКПАО \"Яндекс\".\n   - **Релевантность:** 0.8 (средняя)\n\n### Общая оценка рыночного настроения\n\nНейтральное\n\n### Примечания\n\nДля более детального анализа и актуальных данных рекомендуется обратиться к финансовым новостным ресурсам и пресс-релизам Московской биржи.",
      "influence": {
        "reasoning": "В тексте не содержится значимых новостей, которые могли бы существенно повлиять на цену акций YNDX. Упомянуты только динамика курса акций и информация об условиях их приобретения.",
        "sentiment": "Нейтральное"
      }
    },
}

```


### Развертывание агента в облаке
Если вы хотите развернуть этого агента в облаке и настроить автоматический деплой, можно использовать SourceCraft — платформу для разработки с Git‑репозиториями и CI/CD (сборка, тестирование, развёртывание и сопровождение).
- Документация по **SourceCraft**: https://sourcecraft.dev/portal/docs/ru/
- Пример вызова агента через **SourceCraft**: https://sourcecraft.dev/yandex-cloud-examples/yc-serverless-ai-agent


### Развертывание в Cloud Functions и запуск по расписанию

**Yandex Cloud Functions** — это serverless‑сервис, который позволяет запускать код в виде функций в безопасной, отказоустойчивой и масштабируемой среде без создания и обслуживания виртуальных машин.
Функцию можно вызывать по HTTP‑запросу или через триггеры (например, по событиям или по расписанию).

#### Запуск агента по расписанию
Для периодического запуска агента используйте триггер **Таймер** (Timer): он запускает функцию Cloud Functions по расписанию, заданному cron‑выражением.
Важно: время в cron‑выражении указывается в UTC+0.

Типовой сценарий настройки:
- Создайте функцию Cloud Functions, внутрь которой упакован ваш агент (точка входа — handler).
- Создайте сервисный аккаунт, от имени которого таймер будет вызывать функцию, и выдайте права на вызов (роль `functions.functionInvoker`).
- Создайте триггер типа «Таймер» и привяжите его к функции, указав cron‑расписание.

Документация:
- Cloud Functions (обзор): https://yandex.cloud/ru/docs/functions/
- Обзор сервиса: https://yandex.cloud/ru/docs/functions/concepts/
- Таймер (триггер по расписанию): https://yandex.cloud/ru/docs/functions/concepts/trigger/timer
- Быстрый старт: создание таймера: https://yandex.cloud/ru/docs/functions/quickstart/create-trigger/timer-quickstart
- Пошаговая инструкция: создать таймер: https://yandex.cloud/ru/docs/functions/operations/trigger/timer-create
